# Step 9 : Building Proper Pipeline for Deployment

### Create folders in environment

In [ ]:
Project_btc_sentiment/
  artifacts/
    btc_xgb_rolling_pipeline.joblib
    feature_config.json          (optional)
  data/
    weekly_history.csv
  src/
    features.py
    inference.py
    app_gradio.py

In [1]:
# 1) Check where your notebook is running (CWD) 
import os, sys
print("CWD:", os.getcwd())
# print("First sys.path entries:\n", "\n".join(sys.path[:5]))

CWD: C:\Users\user\ForwardCollege\Project_btc_sentiment\app


In [2]:
# Change the working directory
os.chdir(r"C:\Users\user\ForwardCollege\Project_btc_sentiment\app")

# Verify the change
print("CWD:", os.getcwd())

CWD: C:\Users\user\ForwardCollege\Project_btc_sentiment\app


### Check and verify foders created

In [2]:
import os
os.makedirs("artifacts", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("src", exist_ok=True)
print("Folders ready.")

Folders ready.


### Prepare weekly_history.csv file in appropriate format

In [3]:
# Must have these 6 columns as input
RAW_COLS = ["Sentiment", "Open", "Close", "High", "Low", "Volume"]

In [4]:
import pandas as pd

hist_path = "../app/data/weekly_history.csv"
hist = pd.read_csv(hist_path, index_col=0, parse_dates=True) 
hist = hist.sort_index()

missing = [c for c in RAW_COLS if c not in hist.columns]
print("Missing:", missing)
print("Rows:", len(hist))
print("Columns:", hist.columns.tolist())

Missing: []
Rows: 485
Columns: ['Sentiment', 'Open', 'High', 'Low', 'Close', 'Volume']


### Build the Rolling Feature Transformer and store as python script
- Store this python script as "features.py" in src folder for import later

In [15]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

EPS = 1e-8


def rsi_wilder(close: pd.Series, period: int) -> pd.Series:
    """
    Wilder's RSI implementation (EMA-style smoothing).
    Returns a Series aligned to close.
    """
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)

    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()

    rs = avg_gain / (avg_loss + EPS)
    rsi = 100 - (100 / (1 + rs))
    return rsi


class RollingFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Create engineered features needed by the pretrained XGB model and
    return ONLY the expected features in the exact required order.

    Required raw input columns:
        Close, High, Low, Volume, Sentiment
    """

    def __init__(self, expected_features):
        self.expected_features = list(expected_features)

    def fit(self, X, y=None):
        # No learned params
        return self

    def transform(self, X):
        df = X.copy()

        # 1) Validate raw inputs
        required = ["Close", "High", "Low", "Volume", "Sentiment"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required raw columns: {missing}")

        # 2) Intermediates (NOT returned)
        df["r1"] = df["Close"].pct_change(1)
        df["hl_range"] = (df["High"] - df["Low"]) / (df["Close"] + EPS)

        # 3) Returns / Vol / Mom (only those used by model)
        df["r1_lag2"]  = df["r1"].shift(2)
        df["r1_lag8"]  = df["r1"].shift(8)
        df["r1_lag12"] = df["r1"].shift(12)

        df["r4_lag1"] = df["Close"].pct_change(4).shift(1)

        df["vol_4w_lag1"]  = df["r1"].rolling(4).std().shift(1)
        df["mom_4w_lag1"]  = df["r1"].rolling(4).mean().shift(1)
        df["vol_12w_lag1"] = df["r1"].rolling(12).std().shift(1)
        df["mom_12w_lag1"] = df["r1"].rolling(12).mean().shift(1)

        # 4) HL range stats (std only, 4 and 12)
        df["hl_range_std_4_lag1"]  = df["hl_range"].rolling(4).std().shift(1)
        df["hl_range_std_12_lag1"] = df["hl_range"].rolling(12).std().shift(1)

        # 5) Trend features (MA 4 and 8)
        ma4 = df["Close"].rolling(4).mean()
        ma8 = df["Close"].rolling(8).mean()

        df["close_to_ma_4_lag1"] = ((df["Close"] - ma4) / (ma4 + EPS)).shift(1)
        df["ma_slope_4_lag1"]    = ma4.diff().shift(1)

        df["close_to_ma_8_lag1"] = ((df["Close"] - ma8) / (ma8 + EPS)).shift(1)
        df["ma_slope_8_lag1"]    = ma8.diff().shift(1)

        # 6) Volume features
        df["dollar_volume_lag1"]     = (df["Close"] * df["Volume"]).shift(1)
        df["volume_change_1_lag1"]   = df["Volume"].pct_change().shift(1)

        for w in (4, 8, 12):
            vol_mu = df["Volume"].rolling(w).mean()
            vol_sd = df["Volume"].rolling(w).std()
            df[f"vol_z_{w}_lag1"] = ((df["Volume"] - vol_mu) / (vol_sd + EPS)).shift(1)

        # 7) RSI lag features (4, 8, 12)
        df["RSI_4w_lag1"]  = rsi_wilder(df["Close"], 4).shift(1)
        df["RSI_8w_lag1"]  = rsi_wilder(df["Close"], 8).shift(1)
        df["RSI_12w_lag1"] = rsi_wilder(df["Close"], 12).shift(1)

        # 8) Sentiment base features
        df["sentiment_lag_1"]       = df["Sentiment"].shift(1)
        df["sentiment_chg_1_lag1"]  = df["Sentiment"].diff(1).shift(1)
        df["sentiment_chg_2_lag1"]  = df["Sentiment"].diff(2).shift(1)

        for w in (4, 8):
            df[f"s_mean_{w}_lag1"] = df["Sentiment"].rolling(w).mean().shift(1)
            df[f"s_std_{w}_lag1"]  = df["Sentiment"].rolling(w).std().shift(1)

        mu8 = df["Sentiment"].rolling(8).mean()
        sd8 = df["Sentiment"].rolling(8).std()
        df["sentiment_z8_lag1"] = ((df["Sentiment"] - mu8) / (sd8 + EPS)).shift(1)

        # 9) Interaction features (expected by model)
        df["sentiment_x_momentum_lag1"] = df["sentiment_lag_1"] * df["mom_4w_lag1"]
        df["rsi_x_sentiment_lag1"]      = df["RSI_4w_lag1"] * df["sentiment_lag_1"]
        df["sentiment_x_trend_lag1"]    = df["sentiment_lag_1"] * df["close_to_ma_4_lag1"]

        # 10) LOCK OUTPUT: return exactly expected 33 features in order
        missing_out = [c for c in self.expected_features if c not in df.columns]
        if missing_out:
            raise ValueError(
                "Missing features expected by the pretrained model. "
                f"Examples: {missing_out[:10]} (total missing: {len(missing_out)})"
            )

        return df[self.expected_features].copy()


### Build the pipeline

In [12]:
import joblib
from sklearn.pipeline import Pipeline
from src.features import RollingFeatureEngineer

best_xgb = joblib.load("../app/best_xgb_model.joblib")
expected = best_xgb.feature_names_in_

pipe = Pipeline([
    ("features", RollingFeatureEngineer(expected_features=expected)),
    ("model", best_xgb),
])

# No fit() needed for inference-only usage

In [13]:
# Check whether features name are indentical

import pandas as pd

weekly = pd.read_csv("../app/data/weekly_history.csv", index_col=0, parse_dates=True).sort_index()
X_raw = weekly[["Sentiment","Open","Close","High","Low","Volume"]]  # adjust if your RAW_COLS differs

X_feat = pipe.named_steps["features"].transform(X_raw)

print("Feature shape:", X_feat.shape)
print("Matches expected:", list(X_feat.columns) == list(expected))
print("Missing:", [c for c in expected if c not in X_feat.columns])
print("Extra:", [c for c in X_feat.columns if c not in expected])

Feature shape: (485, 33)
Matches expected: True
Missing: []
Extra: []


### Save the pipeline as artifact

In [14]:
import joblib

joblib.dump(pipe, "../app/artifacts/btc_xgb_rolling_pipeline.joblib")
print("Saved:", "../app/artifacts/btc_xgb_rolling_pipeline.joblibib")

Saved: ../app/artifacts/btc_xgb_rolling_pipeline.joblibib


### Create a safe load_history() + predict_next_week() for gradio app
- Store this in src folder as a "inference.py" for import later

In [8]:
import pandas as pd
import joblib
from src import features

RAW_COLS = ["Sentiment", "Open", "Close", "High", "Low", "Volume"]

PIPE_PATH = "../app/artifacts/btc_xgb_rolling_pipeline.joblib"
HISTORY_PATH = "../app/data/weekly_history.csv"

pipe = joblib.load(PIPE_PATH)

def load_history():
    hist = pd.read_csv(HISTORY_PATH, index_col=0, parse_dates=True)
    hist.sort_index()
    return hist

def predict_next_week(sentiment, open_, close, high, low, volume):
    hist = load_history()

    new_row = pd.DataFrame([{
        "Sentiment": sentiment,
        "Open": open_,
        "Close": close,
        "High": high,
        "Low": low,
        "Volume": volume
    }])

    full = pd.concat([hist[RAW_COLS], new_row], ignore_index=True)

    proba = pipe.predict_proba(full)[-1, 1]   # P(up) for last row
    pred = int(proba >= 0.5)

    return pred, float(proba)

### Testing the prediction before gradio app

#### Load the saved pipeline + sanity checks

In [15]:
import joblib
import pandas as pd
import numpy as np

PIPE_PATH = "../app/artifacts/btc_xgb_rolling_pipeline.joblib"
HIST_PATH = "../app/data/weekly_history.csv"
RAW_COLS  = ["Sentiment", "Open", "Close", "High", "Low", "Volume"]

pipe = joblib.load(PIPE_PATH)

weekly = pd.read_csv(HIST_PATH, index_col=0, parse_dates=True).sort_index()
X_raw  = weekly[RAW_COLS].copy()

# 1) Feature engineering
X_feat = pipe.named_steps["features"].transform(X_raw)

# 2) Drop NaNs like training
valid_idx = X_feat.dropna().index
X_feat_valid = X_feat.loc[valid_idx]

# 3) Predict using the already-fitted model
model = pipe.named_steps["model"]
proba = model.predict_proba(X_feat_valid)
pred  = model.predict(X_feat_valid)

print("Raw rows:", len(X_raw))
print("Valid feature rows after dropna:", len(X_feat_valid))
print("proba shape:", proba.shape)
print("pred counts:", np.unique(pred, return_counts=True))
print("Last prob(up) in valid set:", float(proba[-1, 1]))

Raw rows: 485
Valid feature rows after dropna: 472
proba shape: (472, 2)
pred counts: (array([0, 1]), array([229, 243]))
Last prob(up) in valid set: 0.3567114472389221


In [16]:
last = weekly.iloc[-1]

new_row = pd.DataFrame([{
    "Sentiment": float(last["Sentiment"]),
    "Open": float(last["Open"]),
    "Close": float(last["Close"]) * 1.001,
    "High": float(last["High"]) * 1.001,
    "Low": float(last["Low"]) * 1.001,
    "Volume": float(last["Volume"]),
}], index=[weekly.index.max() + pd.Timedelta(days=7)])

weekly2 = pd.concat([weekly, new_row], axis=0)
X2_raw = weekly2[RAW_COLS]

X2_feat = pipe.named_steps["features"].transform(X2_raw)

# Take only the last row’s features
x_last = X2_feat.iloc[[-1]]

# Guard: if last row has NaNs, you don't have enough history
if x_last.isna().any(axis=1).iloc[0]:
    raise ValueError("Last row features contain NaNs. Need more historical weeks to compute rolling/lag features.")

model = pipe.named_steps["model"]
p_up = float(model.predict_proba(x_last)[0, 1])
yhat = int(model.predict(x_last)[0])

print("Next-week label:", "Uptrend" if yhat == 1 else "Downtrend")
print("Probability Up:", p_up)

Next-week label: Downtrend
Probability Up: 0.43690112233161926
